In [4]:
import numpy as np
import itertools
from typing import Optional
from typing_extensions import Self

### Challenger Puzzle Classes

In [5]:
class Puzzle:
    """
    An object representing the Challenger Puzzle

    :attr prob_array: the problem array
    :attr lower_bound: the lower bound for random number generator
    :attr upper_bound: the upper bound for random number generator
    :attr row_sums: the sum of each row
    :attr col_sums: the sum of each column
    :attr diag_sums: the sum of each diagonal
    :attr _n: the dimension of the puzzle
    """

    def __init__(self: Self, prob_array: np.ndarray, row_sums: np.ndarray, col_sums: np.ndarray, diag_sums: np.ndarray, lower_bound: int, upper_bound: int) -> Self:
        """
        Initializes the puzzle generator object

        :param prob_array: the problem matrix
        :param row_sums: the row sum array
        :param col_sums: the column sum array
        :param diag_sums: the diagonal sum array
        :param lower_bound: the lower bound for cell values
        :param upper_bound: the upper bound for cell values
        """
        if len(prob_array.shape) != 2:
            raise ValueError("problem array must be two-dimensional")
        if prob_array.shape[0] != prob_array.shape[1]:
            raise ValueError("problem array must be square")
        self.prob_array = prob_array
        self._n = prob_array.shape[0]
        self.row_sums = row_sums
        self.col_sums = col_sums
        self.diag_sums = diag_sums
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound

    def __len__(self: Self) -> int:
        """
        Gets the dimension of the Challenger puzzle

        :return: the dimension of the Challenger puzzle
        """
        return self._n

In [6]:
class Generator:
    """
    A Challenger puzzle generator

    :attr n: the dimension of puzzles to generate
    :attr lower_bound: the lower bound of numbers to use for puzzle generation
    :attr upper_bound: the upper bound of numbers to use for puzzle generation
    :attr seed: the optional seed to use for random number generation
    """

    def __init__(self: Self, n: int=4, lower_bound: int=1, upper_bound: int=9, seed: Optional[int]=None) -> None:
        """
        Initializes a puzzle generator

        :param n: the dimension of Challenger puzzles to generate
        :param lower_bound: lower bound for random number generation
        :param upper_bound: upper bound for random number generation
        :param seed: optional seed for random number generation
        """
        self.n = n
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        if seed is not None:
            np.random.seed(seed)

    def generate(self: Self, n: Optional[int]=None, lower_bound: Optional[int]=None, upper_bound: Optional[int]=None, solution: bool=False) -> tuple[Puzzle, Optional[np.ndarray]]:
        """
        Randomly generates a challenger puzzle object

        :param n: optional dimension of puzzle array
        :param lower_bound: optional lower bound for random number generator
        :param upper_bound: optional upper bound for random number generator

        :return: a randomly generated Challenger puzzle and its generating solution (optional)
        """
        if n is None:
            n = self.n
        if lower_bound is None:
            lower_bound = self.lower_bound
        if upper_bound is None:
            upper_bound = self.upper_bound
        gen_array = np.random.randint(lower_bound, upper_bound, size=(n, n))
        available_indices = np.arange(n)
        np.random.shuffle(available_indices)
        prob_array = np.empty((n, n))
        prob_array.fill(np.nan)
        for i in range(n):
            prob_array[i, available_indices[i]] = gen_array[i, available_indices[i]]
        puzzle = Puzzle(
            prob_array,
            np.sum(gen_array, axis=1),
            np.sum(gen_array, axis=0),
            np.array([np.trace(gen_array), np.trace(np.fliplr(gen_array))]),
            lower_bound,
            upper_bound
        )
        return (puzzle, gen_array) if solution else puzzle

In [7]:
class Solver:
    """
    An abstract solver class for Challenger Puzzles

    :attr puzzle: the Challenger Puzzle object for solving
    """

    def __init__(self: Self, puzzle: Puzzle) -> None:
        """
        Initializes a puzzle solver

        :param puzzle: the puzzle to be solved
        """
        self.puzzle = puzzle

    def solve(self: Self) -> object:
        """
        Solves the given Challenger puzzle

        :return: solution(s)
        """
        NotImplemented

In [8]:
class LinalgSolver(Solver):
    """
    A Challenger Puzzle solver that uses Linear Algebra

    :attr puzzle: the Challenger Puzzle object for solving
    """

    def __init__(self: Self, puzzle: Puzzle) -> None:
        """
        Initializes a puzzle solver
        """
        super().__init__(puzzle)

    def solve(self: Self) -> list[np.ndarray]:
        """
        Solves the given Challenger puzzle using linear algebra

        :return: all solutions
        """
        augment_matrix = self.create_augment_matrix()

        rref_matrix = self.row_reduce(augment_matrix)

        ### rref_matrix is the reduced row echelon form matrix.

        free_vars = []

        for j in range(rref_matrix.shape[1]-1):

            leading = False

            for i in range(rref_matrix.shape[0]):
                if rref_matrix[i,j] != 0:
                    if all(rref_matrix[i,k] == 0 for k in range(j)):
                        leading = True

            if not leading:
                free_vars.append(j)

        vals_to_test = list(itertools.product([i for i in range(1,10)], repeat = len(free_vars)))
        solutions = []

        for test in vals_to_test:
            all_vars = [0 for i in range(rref_matrix.shape[1] - 1)]

            test_vec = np.zeros((rref_matrix.shape[1]))
            for index in range(len(test)):
                test_vec[free_vars[index]] = test[index]

            for row in rref_matrix:

                free_vars_total = row @ test_vec
                try:
                    all_vars[np.where(row == 1)[0][0]] = row[-1] - free_vars_total
                except IndexError:
                    pass

            for i in range(len(free_vars)):
                all_vars[free_vars[i]] = test[i]

            if max(all_vars) <= 9 and min(all_vars) >= 1:
                if all(var == var//1 for var in all_vars):
                    solutions.append(all_vars)

        self.num_free_vars = len(free_vars)
        self.num_solutions = len(solutions)
        self.sol_arrays = []

        for sol in solutions:
          counter = 0
          self.sol_arrays.append(self.puzzle.prob_array.copy())
          for index in range(self.puzzle._n**2):
              i = index // self.puzzle._n
              j = index % self.puzzle._n
              if np.isnan(self.sol_arrays[-1][i,j]):
                  self.sol_arrays[-1][i,j] = sol[counter]
                  counter += 1

        return self.sol_arrays

    def row_reduce(self: Self, matrix: np.ndarray) -> np.ndarray:
        """
        Performs row reduction (RREF) on an augmented matrix using NumPy.

        Args:
            matrix (np.ndarray): A 2D numpy array representing the augmented matrix.

        Returns:
            np.ndarray: The RREF of the matrix.
        """
        A = matrix.astype(float).copy()
        rows, cols = A.shape
        pivot_row = 0

        for col in range(cols):
            if pivot_row >= rows:
                break

            # Find the pivot in current column
            pivot = np.argmax(np.abs(A[pivot_row:rows, col])) + pivot_row
            if A[pivot, col] == 0:
                continue  # Cannot pivot on this column

            # Swap current row with pivot row
            A[[pivot_row, pivot]] = A[[pivot, pivot_row]]

            # Normalize pivot row
            A[pivot_row] = A[pivot_row] / A[pivot_row, col]

            # Eliminate column entries in other rows
            for r in range(rows):
                if r != pivot_row:
                    A[r] -= A[r, col] * A[pivot_row]

            pivot_row += 1

        return A

    def create_augment_matrix(self: Self) -> np.ndarray:
        """
        Takes puzzle object and creates augmented matrix that represents it where
        the first n rows are the row sums, the next n rows are the column sums, and
        the next two rows are the diagonal and counter-diagonal, respectively. The
        columns of the augmented matrix are the missing celles in the puzzle from
        left to right, top to bottom.

        Args:
            self: Puzzle objet

        Returns:
            np.ndarray: The augmented matrix
        """
        n = self.puzzle._n
        augment_matrix = np.zeros((2*n + 2, n**2 - n + 1))
        num_of_vars = 0

        for index in range(n**2):
            i = index // n
            j = index % n
            if np.isnan(self.puzzle.prob_array[i,j]):
                augment_matrix[i,num_of_vars] = 1
                augment_matrix[n+j,num_of_vars] = 1

                if i == j:
                    augment_matrix[2*n, num_of_vars] = 1
                if i + j == n - 1:
                    augment_matrix[2*n + 1, num_of_vars] = 1

                num_of_vars += 1

        # Add the last value for each of the row sums
        for index in range(n):
            row = self.puzzle.prob_array[index]

            # Create a boolean mask to identify non-NaN values
            non_nan_mask = ~np.isnan(row)

            augment_matrix[index, n**2 - n] = self.puzzle.row_sums[index] - row[non_nan_mask].item()

        # Add the last value for each of the column sums
        for index in range(n):
            col = self.puzzle.prob_array[:, index]

            # Create a boolean mask to identify non-NaN values
            non_nan_mask = ~np.isnan(col)

            augment_matrix[index + n, n**2 - n] = self.puzzle.col_sums[index] - col[non_nan_mask].item()

        # Add the last value for each of the diagonal sums
        diag_total = 0
        for index in range(n):
            if not(np.isnan(self.puzzle.prob_array[index,index])):
                diag_total += self.puzzle.prob_array[index,index]
        augment_matrix[2*n, n**2 - n] = self.puzzle.diag_sums[0] - diag_total

        # Add the last value for each of the counter-diagonal sums
        cdiag_total = 0
        for index in range(n):
            if not(np.isnan(self.puzzle.prob_array[index, n - index - 1])):
                cdiag_total += self.puzzle.prob_array[index, n - index - 1]
        augment_matrix[2*n + 1, n**2 - n] = self.puzzle.diag_sums[1] - cdiag_total

        return augment_matrix

In [9]:
class MoveSolver(Solver):
    """
    A Challenger Puzzle solver that uses Cameron moves

    :attr puzzle: the Challenger Puzzle object for solving
    """

    def __init__(self: Self, puzzle: Puzzle) -> None:
        """
        Initializes a puzzle solver
        """
        super.__init__(puzzle)

    def solve(self: Self) -> np.ndarray:
        """
        Solves the given Challenger puzzle using Cameron moves

        :return: a single solution
        """
        # Initialize solution
        sol = np.zeros((len(self.puzzle), len(self.puzzle)), dtype=np.int128)

        # Populate diagonal
        for i in range(len(self.puzzle)):
            sol[i, i] = self.puzzle.diag_sums[0] // len(self.puzzle)
        for i in range(self.puzzle.diag_sums[0] % len(self.puzzle)):
            sol[i, i] += 1

        # Populate anti-diagonal
        for i in range(len(self.puzzle)):
            sol[i, len(self.puzzle) - i - 1] = self.puzzle.diag_sums[1] // len(self.puzzle)
        for i in range(self.puzzle.diag_sums[1] % len(self.puzzle)):
            sol[i, len(self.puzzle) - i - 1] += 1

        return sol

### Examples of Code Usage

In [10]:
# Generate random puzzle and view it
my_puzzle = Generator().generate()
print("Problem array:")
print(my_puzzle.prob_array)
print("Row sums:")
print(my_puzzle.row_sums)
print("Col sums:")
print(my_puzzle.col_sums.reshape([-1, 1]))
print("Diag sums:")
print(my_puzzle.diag_sums)

Problem array:
[[ 2. nan nan nan]
 [nan  3. nan nan]
 [nan nan nan  8.]
 [nan nan  4. nan]]
Row sums:
[15 17 20 25]
Col sums:
[[21]
 [20]
 [13]
 [23]]
Diag sums:
[14 23]


In [ ]:
# Solve a puzzle
sol = LinalgSolver(my_puzzle).solve()
print("Number of solutions:", len(sol))
print("One solution:\n", sol[0])

Number of solutions: 18
One solution:
 [[1. 3. 1. 2.]
 [3. 5. 3. 9.]
 [4. 4. 3. 2.]
 [4. 5. 4. 5.]]


### Testing LinAlg Solver

In [ ]:
# Specific puzzle that might break the solver
prob_array = np.array([[np.nan, 1, np.nan, np.nan],
                       [np.nan, np.nan, np.nan, 1],
                       [np.nan, np.nan, 1, np.nan],
                       [1, np.nan, np.nan, np.nan]])
row_sums = np.array([16, 16, 9, 28])
col_sums = np.array([13, 20, 20, 16])
diag_sums = np.array([20, 16])
bad_puzzle = Puzzle(prob_array, row_sums, col_sums, diag_sums, 1, 9)
print("Problem array:")
print(bad_puzzle.prob_array)
print("Row sums:")
print(bad_puzzle.row_sums)
print("Col sums:")
print(bad_puzzle.col_sums.reshape([-1, 1]))
print("Diag sums:")
print(bad_puzzle.diag_sums)
print("")

# Solve it
sol = LinalgSolver(bad_puzzle).solve()
print("Number of solutions:", len(sol))
print("First solution:\n", sol[0])

Problem array:
[[nan  1. nan nan]
 [nan nan nan  1.]
 [nan nan  1. nan]
 [ 1. nan nan nan]]
Row sums:
[16 16  9 28]
Col sums:
[[13]
 [20]
 [20]
 [16]]
Diag sums:
[20 16]

Number of solutions: 0


IndexError: list index out of range